# 03 Compuertas de calidad y deduplicación

Bronze conserva los mensajes. Silver exige MMSI de nueve dígitos distinto de cero, tiempo válido, coordenadas finitas en rango, posición distinta de (0, 0), SOG en [0, 60] nudos y COG en [0, 360) grados. `COG = 360` es el centinela de rumbo no disponible. Heading, eslora y manga pueden faltar. Una fila inválida se conserva con todos sus motivos.

A propósito no hay chequeo de tierra contra mar. El dato NOAA incluye puertos, ríos y los Grandes Lagos, y un mapa de tierra grueso marcaría tráfico real como si estuviera en tierra.

**Por qué MMSI y BaseDateTime.** El MMSI identifica el emisor y el tiempo UTC identifica su observación. Agregar latitud o longitud a la llave retendría dos posiciones incompatibles para el mismo instante y distorsionaría la distancia en oro. Usar solo MMSI descartaría la trayectoria. Primero se evalúa cada copia para que una copia inválida no desplace a la válida. Dentro del batch se conserva una copia válida por `(MMSI, BaseDateTime)`. Si la clave ya fue aceptada en un batch anterior, se conserva la primera versión.

Cada batch anota en `oceanwatch_metricas_silver` cuántas filas entraron, cuántas son nuevas, cuántas se repiten dentro del batch, cuántas ya estaban y cuántas van a cuarentena. `batch_alta` queda en la fila nueva. Gold lo usa para reabrir un día cuando un batch posterior trae una posición de ese día. El checkpoint `silver_ais` está vacío la primera vez y relee bronze.

| motivo | Condición | Por qué |
|---|---|---|
| `mmsi_invalido` | `MMSI` nulo, no tiene nueve dígitos, o es `000000000` | Sin buque no hay trayectoria. |
| `sin_timestamp` | `BaseDateTime` nulo | Sin instante no hay `fecha_dia` ni clave del `MERGE`. |
| `latitud_invalida` | `LAT` nula, NaN, o fuera de [-90, 90] | No es una posición para haversine ni para H3. |
| `longitud_invalida` | `LON` nula, NaN, o fuera de [-180, 180] | Igual que la latitud. |
| `posicion_cero` | `LAT` y `LON` son exactamente 0 | El par (0, 0) es un centinela, no una posición. Un cero en un solo eje no basta: el ecuador y el meridiano de Greenwich son coordenadas reales. |
| `sog_invalido` | `SOG` nulo, NaN, o fuera de [0, 60] | Incluye el centinela ITU `102.3`. Gold calcula `sog_media` con estas filas. |
| `cog_invalido` | `COG` nulo, NaN, o fuera de [0, 360) | `COG = 360` es rumbo no disponible y va a cuarentena. |

`Heading = 511` se queda, con su valor. Es rumbo ausente, no una posición mala.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))


In [ ]:
%run ./_config


In [ ]:
from pyspark.sql import functions as F

COLUMNAS = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
    "_source_file", "_ingested_at",
]


def con_motivo(df):
    """Marca las reglas fallidas. concat_ws omite los null, asi que '' significa que la fila es valida."""
    lat, lon = F.col("LAT"), F.col("LON")
    lat_ok = lat.between(-90, 90) & ~F.isnan(lat)
    lon_ok = lon.between(-180, 180) & ~F.isnan(lon)
    mmsi = F.col("MMSI")
    return df.withColumn("motivo", F.concat_ws(
        ",",
        F.when(mmsi.isNull() | ~mmsi.rlike("^[0-9]{9}$") | (mmsi == "000000000"), "mmsi_invalido"),
        F.when(F.col("BaseDateTime").isNull(), "sin_timestamp"),
        F.when(lat.isNull() | ~lat_ok, "latitud_invalida"),
        F.when(lon.isNull() | ~lon_ok, "longitud_invalida"),
        F.when((lat == 0) & (lon == 0), "posicion_cero"),
        F.when(F.col("SOG").isNull() | ~F.col("SOG").between(0, 60) | F.isnan("SOG"), "sog_invalido"),
        F.when(F.col("COG").isNull() | ~F.col("COG").between(0, 360) | (F.col("COG") == 360) | F.isnan("COG"), "cog_invalido"),
    ))


def escribir_batch(df, tabla, app, batch_id):
    (
        df.write.format("delta")
        .mode("append")
        .option("txnAppId", app)
        .option("txnVersion", int(batch_id))
        .saveAsTable(tabla)
    )


def aplicar_silver(batch_df, batch_id):
    sesion = batch_df.sparkSession
    marcado = con_motivo(batch_df)
    n_entrada = marcado.count()
    invalidos = marcado.where(F.col("motivo") != "")
    validos = (
        marcado.where(F.col("motivo") == "")
        .drop("motivo")
        .dropDuplicates(["MMSI", "BaseDateTime"])
        .withColumn("fecha_dia", F.to_date("BaseDateTime"))
        .withColumn("batch_alta", F.lit(int(batch_id)).cast("long"))
    )
    n_cuarentena = invalidos.count()
    n_candidatas = validos.count()
    app = "oceanwatch-silver"
    if n_cuarentena:
        escribir_batch(invalidos.select(*COLUMNAS, "motivo"), CUARENTENA, app + "-rechazos", batch_id)
    nuevas = 0
    if n_candidatas:
        fechas = [
            fila.fecha_dia.isoformat()
            for fila in validos.select("fecha_dia").distinct().collect()
            if fila.fecha_dia is not None
        ]
        if fechas:
            lista = ", ".join(f"DATE '{dia}'" for dia in fechas)
            validos.createOrReplaceTempView("oceanwatch_silver_batch")
            sesion.sql(f"""
                MERGE INTO {SILVER} AS destino
                USING oceanwatch_silver_batch AS origen
                ON destino.fecha_dia IN ({lista})
               AND destino.MMSI = origen.MMSI
               AND destino.BaseDateTime = origen.BaseDateTime
                WHEN NOT MATCHED THEN INSERT *
            """)
            sesion.catalog.dropTempView("oceanwatch_silver_batch")
            nuevas = int(sesion.sql(
                f"SELECT count(*) AS n FROM {SILVER} "
                f"WHERE fecha_dia IN ({lista}) AND batch_alta = {int(batch_id)}"
            ).first()["n"])
    intra = n_entrada - n_cuarentena - n_candidatas
    previos = n_candidatas - nuevas
    metricas = sesion.sql(f"""
        SELECT
          {int(batch_id)}L AS batch,
          DATE '{fecha}' AS fecha_job,
          {n_entrada}L AS entrada,
          {nuevas}L AS nuevas,
          {intra}L AS duplicados_batch,
          {previos}L AS duplicados_previos,
          {n_cuarentena}L AS cuarentena,
          current_timestamp() AS ejecutado_en
    """)
    escribir_batch(metricas, METRICAS_SILVER, app + "-metricas", batch_id)
    print(
        f"batch {batch_id}: entrada={n_entrada} nuevas={nuevas} "
        f"duplicados_batch={intra} duplicados_previos={previos} cuarentena={n_cuarentena}"
    )


## Ejecución

El stream lee bronze con el checkpoint `silver_ais`. La primera ejecución relee el histórico. Las siguientes solo ven lo que bronze aún no había entregado.


In [ ]:
for consulta in list(spark.streams.active):
    if consulta.name == "oceanwatch_silver_ais":
        consulta.stop()

consulta = (
    spark.readStream.format("delta")
    .table(BRONZE)
    .writeStream.foreachBatch(aplicar_silver)
    .queryName("oceanwatch_silver_ais")
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
print(f"Ejecución silver {fecha_dia.isoformat()} terminada.")
